In [1]:
from unike.utils import Link
from unike.module.model import TransE
import pandas as pd

In [2]:
ent_tol = 121649
rel_tol = 22

In [3]:
model = TransE(
	ent_tol = ent_tol,
	rel_tol = rel_tol,
	dim = 100, 
	p_norm = 1,
	norm_flag = True
)

In [12]:
model.load_checkpoint('checkpoints/transe/all/multi/transe-300.pth')

In [5]:
link = Link(
    in_path='data',
    model=model
)

In [6]:
all_drug_ent_id = [link.ent2id[ent_name] for ent_name in link.ent2id.keys() if ent_name.split(':')[-1] == 'drug']
len(all_drug_ent_id)

9542

In [7]:
nodes = pd.read_csv('../../../kg/nodes.csv')

In [8]:
dmd_approved_triples_index = pd.read_csv('../approved_triples_latest_index.csv')
dmd_approved_triples_index

,relation,x_index,y_index,uid
0,disease_phenotype_positive,40189,27374,:40400011:8cxiHTfHEbuMJ5cudTfhtn
1,disease_protein,40189,7812,:40366239:52d5FmoktMw9P8H9yRsyEL
2,disease_protein,40189,4785,:40361216:QW9DbX3U4yc8iP2ugNLKfx
3,disease_phenotype_positive,40189,28022,:40359883:JrogmgpD3XR4BMnqfQWogW
4,disease_protein,40189,290,:40195304:gE7Xu5hLF966xvk6Ez4Xmj
...,...,...,...,...
292,bioprocess_protein,57069,2241,:32157826:fWGA3LyQgfLJ9QmjrXMqEk
293,bioprocess_protein,57069,8107,:32157826:gNvwG4tWw29KWF2h5ktDK2
294,bioprocess_protein,57069,13265,:32157826:FV7or8bQdgswq2MBBanUBM
295,bioprocess_protein,57069,2884,:32157826:nsjnBYuNefSTxvMYywN49o


In [9]:

def is_llm_ext(df, relation: str):
    dmd_approved_triples_index_copy = dmd_approved_triples_index.query(f'relation == "{relation}"')
    dmd_approved_triples_index_copy.astype({'x_index': int}).astype({'y_index': int})
    df.astype({'head': int}).astype({'tail': int})
    df_merged = pd.merge(df, dmd_approved_triples_index_copy[['x_index', 'y_index', 'uid']], left_on=['head', 'tail'], right_on=['x_index', 'y_index'], how='left')
    df_merged = df_merged.drop(columns=['x_index', 'y_index'])  # 保留uid
    print(df_merged.head())
    return df_merged

In [10]:
df = link.link(all_drug_ent_id, [3], [40189], device='cuda:0').drop(columns=['score'])
df = pd.merge(df, nodes, left_on=['head'], right_on=['node_index'], how='left').drop(columns=['node_index', 'node_type', 'node_name']).rename(columns={'node_id': 'head_id', 'node_source': 'head_source'})
df = is_llm_ext(df, 'indication')
df.to_csv('link_result_transe_400_DMD.csv', index=False)

    head  rel   tail     in          head_ent     rel_ent  \
0  24971    3  40189   True   Eteplirsen:drug  indication   
1  24986    3  40189   True  Viltolarsen:drug  indication   
2  17852    3  40189  False      Quinine:drug  indication   
3  25793    3  40189  False   Guacetisal:drug  indication   
4  24465    3  40189   True   Golodirsen:drug  indication   

                              tail_ent        head_id head_source  uid  
0  Duchenne muscular dystrophy:disease  kg4rd:DB06014    DrugBank  NaN  
1  Duchenne muscular dystrophy:disease  kg4rd:DB15005    DrugBank  NaN  
2  Duchenne muscular dystrophy:disease  kg4rd:DB00468    DrugBank  NaN  
3  Duchenne muscular dystrophy:disease  kg4rd:DB13538    DrugBank  NaN  
4  Duchenne muscular dystrophy:disease  kg4rd:DB15593    DrugBank  NaN  


In [13]:
df = link.link(all_drug_ent_id, [3], [49932], device='cuda:0').drop(columns=['score'])
df = pd.merge(df, nodes, left_on=['head'], right_on=['node_index'], how='left').drop(columns=['node_index', 'node_type', 'node_name']).rename(columns={'node_id': 'head_id', 'node_source': 'head_source'})
df = is_llm_ext(df, 'indication')
df.to_csv('link_result_transe_300_AD.csv', index=False)

    head  rel   tail     in           head_ent     rel_ent  \
0  19746    3  49932  False     Metformin:drug  indication   
1  17148    3  49932  False  Aripiprazole:drug  indication   
2  16946    3  49932  False    Olanzapine:drug  indication   
3  17211    3  49932  False     Asenapine:drug  indication   
4  20597    3  49932  False   Pramlintide:drug  indication   

                    tail_ent        head_id head_source  uid  
0  Alzheimer disease:disease  kg4rd:DB00331    DrugBank  NaN  
1  Alzheimer disease:disease  kg4rd:DB01238    DrugBank  NaN  
2  Alzheimer disease:disease  kg4rd:DB00334    DrugBank  NaN  
3  Alzheimer disease:disease  kg4rd:DB06216    DrugBank  NaN  
4  Alzheimer disease:disease  kg4rd:DB01278    DrugBank  NaN  


In [11]:
df = link.link(all_drug_ent_id, [1], [9662], device='cuda:0').drop(columns=['score'])
df = pd.merge(df, nodes, left_on=['head'], right_on=['node_index'], how='left').drop(columns=['node_index', 'node_type', 'node_name']).rename(columns={'node_id': 'head_id', 'node_source': 'head_source'})
df = is_llm_ext(df, 'drug_protein')
df.to_csv('link_result_transe_400_TAK1.csv', index=False)

    head  rel  tail     in             head_ent       rel_ent  \
0  18574    1  9662  False    Fostamatinib:drug  drug_protein   
1  17080    1  9662  False      Methyldopa:drug  drug_protein   
2  17751    1  9662  False     Terbutaline:drug  drug_protein   
3  21113    1  9662  False  Trichostatin A:drug  drug_protein   
4  17755    1  9662  False    Procainamide:drug  drug_protein   

              tail_ent        head_id head_source  uid  
0  MAP3K7:gene/protein  kg4rd:DB12010    DrugBank  NaN  
1  MAP3K7:gene/protein  kg4rd:DB00968    DrugBank  NaN  
2  MAP3K7:gene/protein  kg4rd:DB00871    DrugBank  NaN  
3  MAP3K7:gene/protein  kg4rd:DB04297    DrugBank  NaN  
4  MAP3K7:gene/protein  kg4rd:DB01035    DrugBank  NaN  


In [12]:
df = link.link(all_drug_ent_id, [1], [5153], device='cuda:0').drop(columns=['score'])
df = pd.merge(df, nodes, left_on=['head'], right_on=['node_index'], how='left').drop(columns=['node_index', 'node_type', 'node_name']).rename(columns={'node_id': 'head_id', 'node_source': 'head_source'})
df = is_llm_ext(df, 'drug_protein')
df.to_csv('link_result_transe_400_S1PR1.csv', index=False)

    head  rel  tail     in               head_ent       rel_ent  \
0  24404    1  5153   True       Ceralifimod:drug  drug_protein   
1  23402    1  5153   True          ASP-4058:drug  drug_protein   
2  18010    1  5153  False  Dextromethorphan:drug  drug_protein   
3  17950    1  5153   True         Ponesimod:drug  drug_protein   
4  18567    1  5153  False        Esketamine:drug  drug_protein   

             tail_ent        head_id head_source  uid  
0  S1PR1:gene/protein  kg4rd:DB16123    DrugBank  NaN  
1  S1PR1:gene/protein  kg4rd:DB11819    DrugBank  NaN  
2  S1PR1:gene/protein  kg4rd:DB00514    DrugBank  NaN  
3  S1PR1:gene/protein  kg4rd:DB12016    DrugBank  NaN  
4  S1PR1:gene/protein  kg4rd:DB11823    DrugBank  NaN  


In [14]:
df_tak1_head = pd.read_csv('link_result_transe_400_TAK1.csv')
df_s1pr1_head = pd.read_csv('link_result_transe_400_S1PR1.csv')

In [15]:
tak1_set = set(df_tak1_head['head_id'].head(100).to_list())
print(len(tak1_set))
s1pr1_set = set(df_s1pr1_head['head_id'].head(100).to_list())
print(len(s1pr1_set))
print(tak1_set & s1pr1_set)

100
100
{'kg4rd:DB12645', 'kg4rd:DB00619', 'kg4rd:DB13146', 'kg4rd:DB12612', 'kg4rd:DB00836', 'kg4rd:DB00668'}
